In [12]:
import os
from transformers import AutoTokenizer

MODEL = "/kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1"
if not os.path.exists(MODEL):
    MODEL = "xlm-roberta-base"

tok = AutoTokenizer.from_pretrained(MODEL)


In [13]:
import torch
print(torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

import os
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

MODEL = "/kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1"
if not os.path.exists(MODEL):
    MODEL = "xlm-roberta-base"

DATA = "/kaggle/input/competitions/rmit-hackathon-2026"

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {l: i for i, l in enumerate(LABELS)}

MAXLEN, EPOCHS, LR = 256, 1, 2e-5
BS_PER_GPU = 16
NGPU = max(torch.cuda.device_count(), 1)

train = pd.read_csv("train.csv")[:200]
test = pd.read_csv("test.csv")

tr, va = train_test_split(train, test_size=0.1, stratify=train.label, random_state=42)

tok = AutoTokenizer.from_pretrained(MODEL)

def mk(df, labelled=True):
    d = df[["text"]].copy()
    if labelled:
        d["labels"] = df.label.map(L2I).values
    return (
        Dataset.from_pandas(d, preserve_index=False)
        .map(lambda b: tok(b["text"], truncation=True, max_length=MAXLEN), batched=True, remove_columns=["text"])
    )

ds_tr, ds_va, ds_te = mk(tr), mk(va), mk(test, labelled=False)
print(len(ds_tr), len(ds_va), len(ds_te))


1 ['NVIDIA GeForce RTX 5050 Laptop GPU']


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

180 20 6000


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, 
    num_labels=len(LABELS),
    id2label={i: l for l, i in L2I.items()}, 
    label2id=L2I
)

print(model)

c:\python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rrana\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            